In [1]:
!pip install pymupdf sentence-transformers


[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import fitz
import os
import json
from sentence_transformers import SentenceTransformer


load pdf


In [3]:
pdf_path = "digitalbook.pdf"
doc=fitz.open(pdf_path)
print ("Total pages:", len(doc))


Total pages: 565


In [4]:
pages = []
for page_number,page in enumerate(doc):
    text = page.get_text()
    pages.append({"page_number": page_number, "text": text})    
print(pages[0])  # Print the first page's content


{'page_number': 0, 'text': ''}


TEXT CLEANING

In [5]:
def clean_text(text):
    # Remove newlines and extra spaces
     text = text.replace('\n', ' ')
     text = ' '.join(text.split())
   
     return text

In [6]:
for page in pages:
    page['text'] = clean_text(page['text'])

print(pages[4])  # Print the cleaned first page's content




{'page_number': 4, 'text': 'Digital Design With an Introduction to the Verilog HDL FIFTH EDITION M. Morris Mano Emeritus Professor of Computer Engineering California State University, Los Angeles Michael D. Ciletti Emeritus Professor of Electrical and Computer Engineering University of Colorado at Colorado Springs Upper Saddle River Boston Columbus San Franciso New York Indianapolis London Toronto Sydney Singapore Tokyo Montreal Dubai Madrid Hong Kong Mexico City Munich Paris Amsterdam Cape Town'}


CHUNKING

In [7]:
def chunk_code(text, chunk_size=2000, overlap=200):
    """
    Splits the text into chunks of specified size with a specified overlap.
    
    Parameters:
    - text (str): The input text to be chunked.
    - chunk_size (int): The maximum size of each chunk.
    - overlap (int): The number of overlapping characters between chunks.
    
    Returns:
    - List[str]: A list of text chunks.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap  # Move start forward by chunk_size minus overlap
    return chunks


In [8]:
chunks = []
for page in pages:
    page_chunks = chunk_code(page['text'])
    for chunk in page_chunks:
        chunks.append({"page_number": page['page_number'], "chunk": chunk})

print("tottal chunks:", len(chunks))
print(chunks[3])  # Print the first chunk's content

tottal chunks: 906
{'page_number': 5, 'chunk': "Library of Congress Cataloging-in-Publication Data Mano, M. Morris, 1927– Digital design : with an introduction to the verilog hdl / M. Morris Mano, Michael D. Ciletti.—5th ed. p. cm. Includes index. ISBN-13: 978-0-13-277420-8 ISBN-10: 0-13-277420-8 1. Electronic digital computers—Circuits. 2. Logic circuits. 3. Logic design. 4. Digital integrated circuits. I. Ciletti, Michael D. II. Title. TK7888.3.M343 2011 621.39'5—dc23 2011039094 Vice President and Editorial Director, ECS: Marcia J. Horton Executive Editor: Andrew Gilfillan Vice-President, Production: Vince O’Brien Executive Marketing Manager: Tim Galligan Marketing Assistant: Jon Bryant Permissions Project Manager: Karen Sanatar Senior Managing Editor: Scott Disanno Production Project Manager/Editorial Production Manager: Greg Dulles Cover Designer: Jayne Conte Cover Photo: Michael D. Ciletti Composition: Jouve India Private Limited Full-Service Project Management: Jouve India Privat

EMBEDDING

In [9]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
texts = [chunk['chunk'] for chunk in chunks]
embeddings = model.encode(texts,show_progress_bar=True)


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

In [11]:
print("Embeddings shape:", embeddings.shape)  # Print the shape of the embeddings


Embeddings shape: (906, 384)


In [12]:
for i, chunk in enumerate(chunks):
    chunk['embedding'] = embeddings[i].tolist()  # Convert numpy array to list for JSON serialization

SAVING THE PROCESSED DATA

In [13]:
with open('chunks_with_embeddings.json', 'w') as f:
    json.dump(chunks, f, indent=4)

STORE EMBEDDING IN VECTORED DATABASE

In [14]:
!pip install faiss-cpu


[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import faiss
import numpy as np

In [16]:
embedding_matrix = np.array([chunk['embedding'] for chunk in chunks]).astype('float32')
print("Embedding matrix shape:", embedding_matrix.shape)  # Print the shape of the embedding matrix

Embedding matrix shape: (906, 384)


In [17]:
dim = embedding_matrix.shape[1]  # Dimension of the embeddings
index = faiss.IndexFlatL2(dim)  # Create a FAISS index for L2 distance
index.add(embedding_matrix)  # Add the embeddings to the index  
print("Number of vectors in the index:", index.ntotal)  # Print the number of vectors in the index

Number of vectors in the index: 906


SEMANTIC SEARCH


In [18]:
def semantic_search(query, top_k=5):
    """
    Performs semantic search on the indexed embeddings using the provided query.
    
    Parameters:
    - query (str): The input query string.
    - top_k (int): The number of top results to return.
    
    Returns:
    - List[Dict]: A list of dictionaries containing the page number, chunk, and distance for each result.
    """
    query_embedding = model.encode([query]).astype('float32')  # Encode the query
    distances, indices = index.search(query_embedding, top_k)  # Search the index
    results = []
    for i in range(top_k):
        idx = indices[0][i]
        distance = distances[0][i]
        results.append({
            "page_number": chunks[idx]['page_number'],
            "chunk": chunks[idx]['chunk'],
            "distance": float(distance)
        })
    return results    

In [19]:
from sentence_transformers import SentenceTransformer

test_query = "What is the main topic of the book?"
# Ensure the embedding model is available before running semantic search
if 'model' not in globals():
    model = SentenceTransformer('all-MiniLM-L6-v2')

if 'index' not in globals():
    if 'embedding_matrix' not in globals():
        embedding_matrix = np.asarray(
            [chunk['embedding'] for chunk in chunks],
            dtype='float32'
        )
    index = faiss.IndexFlatL2(embedding_matrix.shape[1])
    index.add(embedding_matrix)

results = semantic_search(test_query, top_k=5)
for result in results:
    print(f"Page Number: {result['page_number']}, Distance: {result['distance']}")
    print(f"Chunk: {result['chunk']}\n")    

Page Number: 85, Distance: 1.29807448387146
Chunk: veral exercises and design problems based on Verilog throughout the book.

Page Number: 366, Distance: 1.5760371685028076
Chunk: Design. Upper Saddle River, NJ: Prentice Hall.

Page Number: 128, Distance: 1.6590079069137573
Chunk: t is a comment (optional) providing useful information to the reader. The second line begins with the keyword module and starts the declaration (description) of the module; the last line completes the declaration with the keyword endmodule. The keyword module is followed by a name and a list of ports. The name ( Simple_Circuit in this example) is an identifier. Identifiers are names given to modules, variables (e.g., a signal), and other elements of the language so that they can be referenced in the design. In general, we choose meaningful names for modules. Identifiers are composed of alpha- numeric characters and the underscore (_), and are case sensitive. Identifiers must start with an alphabetic character

In [20]:
!pip install rank-bm25


[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
from rank_bm25 import BM25Okapi

In [22]:
corpus = [chunk['chunk'] for chunk in chunks]
bm25 = BM25Okapi([doc.split() for doc in corpus])
query = "What is the main topic of the book?"
scores = bm25.get_scores(query.split())
top_indices = scores.argsort()[-5:][::-1]  # Get indices of top 5 scores in descending order
for idx in top_indices:
    print(f"Page Number: {chunks[idx]['page_number']}, Score: {scores[idx]}")
    print(f"Chunk: {chunks[idx]['chunk']}\n")   

Page Number: 266, Score: 20.98471399179048
Chunk: Problems 249 (a)* Use D flip-flops in the design. (b) Use JK flip-flops in the design. 001 010 011 100 000 1/1 1/1 1/0 1/1 1/1 0/0 0/0 0/0 0/0 0/0 FIGURE P5.19 5.20 Design the sequential circuit specified by the state diagram of Fig. 5.19 , using T flip-flops. 5.21 What is the main difference between an initial statement and an always statement in Verilog HDL? 5.22 Draw the waveform generated by the statements below: (a) initial begin w = 0; #10 w = 1; # 40 w = 0; # 20 w = 1; #15 w = 0; end (b) initial fork w = 0; #10 w = 1; # 40 w = 0; # 20 w = 1; #15 w = 0; join 5.23* Consider the following statements assuming that RegA contains the value of 50 initially. (a) RegA = 125; (b) RegA <= 125; RegB = RegA; RegB <= RegA; What are the values of RegA and RegB after execution? 5.24 Write and verify an HDL behavioral description of a positive-edge-sensitive D flip-flop with asynchronous preset and clear. 5.25 A special positive-edge-triggered fl

In [23]:
# Map semantic results back to their chunk indices
semantic_rank = {}
for rank, item in enumerate(results, start=1):
    idx = next(
        i for i, chunk in enumerate(chunks)
        if chunk["page_number"] == item["page_number"]
        and chunk["chunk"] == item["chunk"]
    )
    semantic_rank[idx] = rank

bm25_rank = {int(idx): rank for rank, idx in enumerate(top_indices, start=1)}

# Reciprocal rank fusion; lower ranks are better
candidate_indices = set(semantic_rank) | set(bm25_rank)
final_scores = {
    idx: 1 / (60 + semantic_rank.get(idx, 1000))
       + 1 / (60 + bm25_rank.get(idx, 1000))
    for idx in candidate_indices
}

for idx in sorted(candidate_indices, key=final_scores.get, reverse=True)[:5]:
    print(f"Page Number: {chunks[idx]['page_number']}")
    print(f"Chunk: {chunks[idx]['chunk']}\n")

print("Final Scores:", final_scores)  # Print the final scores for debugging

Page Number: 85
Chunk: veral exercises and design problems based on Verilog throughout the book.

Page Number: 266
Chunk: Problems 249 (a)* Use D flip-flops in the design. (b) Use JK flip-flops in the design. 001 010 011 100 000 1/1 1/1 1/0 1/1 1/1 0/0 0/0 0/0 0/0 0/0 FIGURE P5.19 5.20 Design the sequential circuit specified by the state diagram of Fig. 5.19 , using T flip-flops. 5.21 What is the main difference between an initial statement and an always statement in Verilog HDL? 5.22 Draw the waveform generated by the statements below: (a) initial begin w = 0; #10 w = 1; # 40 w = 0; # 20 w = 1; #15 w = 0; end (b) initial fork w = 0; #10 w = 1; # 40 w = 0; # 20 w = 1; #15 w = 0; join 5.23* Consider the following statements assuming that RegA contains the value of 50 initially. (a) RegA = 125; (b) RegA <= 125; RegB = RegA; RegB <= RegA; What are the values of RegA and RegB after execution? 5.24 Write and verify an HDL behavioral description of a positive-edge-sensitive D flip-flop with 

In [24]:
from sentence_transformers import cross_encoder

# Initialize the cross-encoder for semantic rankingre
reranker = cross_encoder.CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [25]:
pair = [(test_query, chunks[idx]['chunk'])
for idx in sorted(candidate_indices, key=final_scores.get, reverse=True)[:5]]

In [26]:
rank_results = sorted(
    [(idx, score) for idx, score in zip(
        [idx for idx, _ in pair],
        reranker.predict(pair)
    )], key=lambda x: x[1], reverse=True
)

In [27]:
!pip install ollama
import ollama


[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {
            "role": "user",
            "content": "Explain RAG in simple terms."
        }
    ]
)

print(response["message"]["content"])

RAG is a simple concept that stands for:

**R** - Red
**A** - Amber
**G** - Green

It's often used in project management, IT, and other fields to indicate the status of a task or a system. Here's what each color typically means:

* **Red (R)**: Something is broken or not working as expected. It's an alert that requires immediate attention.
* **Amber (A)**: There's a problem or a warning, but it's not critical. It's a signal that something needs to be checked or fixed, but it's not urgent.
* **Green (G)**: Everything is working fine and as expected. It's a sign that the task or system is functioning properly.

Think of it like traffic lights:

* Red means "stop" ( something's broken)
* Amber means "slow down" (something's not quite right)
* Green means "go" (everything's good to go)


In [29]:
import ollama

response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {
            "role": "user",
            "content": "In AI, what does RAG stand for? Explain Retrieval-Augmented Generation in 2 simple sentences."
        }
    ]
)

print(response["message"]["content"])

In AI, RAG stands for Retrieval-Augmented Generation. Retrieval-Augmented Generation is a type of AI model that combines the strengths of retrieval-based and generation-based approaches to generate text. It works by first retrieving relevant context from a large knowledge base or database, and then using this context to inform and augment a generation model, which produces a final output.


In [30]:
print(rank_results[:2])   
top_results = rank_results[:5]

[('What is the main topic of the book?', np.float32(-8.12946)), ('What is the main topic of the book?', np.float32(-10.691016))]


In [31]:
candidate_order = sorted(candidate_indices, key=final_scores.get, reverse=True)[:5]
rerank_scores = reranker.predict(pair)

top_results = sorted(
    zip(candidate_order, rerank_scores),
    key=lambda item: item[1],
    reverse=True
)

context_parts = []

for idx, score in top_results:
    chunk = chunks[idx]
    context_parts.append(
        f"[Page {chunk['page_number']}]\n{chunk['chunk']}"
    )

context = "\n\n".join(context_parts)

print(context)

[Page 85]
veral exercises and design problems based on Verilog throughout the book.

[Page 489]
472 Chapter 9 Laboratory Experiments bidirectional shift register. The rate at which the light moves is determined by the fre- quency of the clock. The circuit is first initialized with the reset switch. The start switch starts the game by placing the ball (an indicator lamp) at the extreme right. The player must press the pulser push button to start the ball moving to the left. The single light shifts to the left until it reaches the leftmost position (the wall), at which time the ball returns to the player by reversing the direction of shift of the moving light. When the light is again at the rightmost position, the player must press the pulser again to reverse the direction of shift. If the player presses the pulser too soon or too late, the ball dis- appears and the light goes off. The game can be restarted by turning the start switch on and then off. The start switch must be open (logic

In [32]:
question = "What are the major findings of this document?"
retrieved_chunks = [
    chunks[idx]["chunk"]
    for idx, _ in top_results
]

context = "\n\n".join(retrieved_chunks)

prompt = f"""
You are a question-answering assistant for a digital electronics textbook.

Answer the user's question using ONLY the information provided in the context below.

If the answer is present in the context, answer it directly and clearly.
Do not say that the information is unavailable if the context contains the answer.
If the context genuinely does not contain the answer, then say:
"Information not found in the provided document."

Context:
{context}

User Question:
{question}

Answer:
"""

In [ ]:
import time

try:
    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "num_ctx": 4096,
            "num_predict": 512,
            "temperature": 0.2
        },
        keep_alive="5m"
    )

    answer = response["message"]["content"]
    print(answer)

except Exception as error:
    print("Ollama request failed:", error)
    print("Ensure Ollama is running and the model is available.")

ResponseError: an error was encountered while running the model: read tcp 127.0.0.1:65419->127.0.0.1:56640: wsarecv: An existing connection was forcibly closed by the remote host. (status code: 500)

In [ ]:
answer = response["message"]["content"]
print(answer)

Unfortunately, the information is not available in the document. The document appears to be a collection of various chapters and sections, covering topics such as digital logic design, Verilog programming, and laboratory experiments. It does not provide a summary of major findings.

However, I can provide information on specific topics, such as the function of a reset switch (Page 489), the operation of a bidirectional shift register (Page 489), or the design of sequential circuits using D flip-flops, JK flip-flops, and T flip-flops (Pages 266-366).


In [ ]:
question = "what is a reset switch?"
print(f"User Question: {question}")
print(f"Answer: {answer}")

User Question: what is a reset switch?
Answer: Unfortunately, the information is not available in the document. The document appears to be a collection of various chapters and sections, covering topics such as digital logic design, Verilog programming, and laboratory experiments. It does not provide a summary of major findings.

However, I can provide information on specific topics, such as the function of a reset switch (Page 489), the operation of a bidirectional shift register (Page 489), or the design of sequential circuits using D flip-flops, JK flip-flops, and T flip-flops (Pages 266-366).


In [ ]:
print("USER QUESTION:")
print(question)

print("\nRETRIEVED CHUNKS:")
retrieved_chunks = [chunks[idx]["chunk"] for idx, _ in top_results]
for i, chunk in enumerate(retrieved_chunks):
    print(f"\n--- CHUNK {i+1} ---")
    print(chunk)

USER QUESTION:
what is a reset switch?

RETRIEVED CHUNKS:

--- CHUNK 1 ---
veral exercises and design problems based on Verilog throughout the book.

--- CHUNK 2 ---
472 Chapter 9 Laboratory Experiments bidirectional shift register. The rate at which the light moves is determined by the fre- quency of the clock. The circuit is first initialized with the reset switch. The start switch starts the game by placing the ball (an indicator lamp) at the extreme right. The player must press the pulser push button to start the ball moving to the left. The single light shifts to the left until it reaches the leftmost position (the wall), at which time the ball returns to the player by reversing the direction of shift of the moving light. When the light is again at the rightmost position, the player must press the pulser again to reverse the direction of shift. If the player presses the pulser too soon or too late, the ball dis- appears and the light goes off. The game can be restarted by turning 